In [13]:
# ===================================================
# Raw data : SCHARE_Primary_Care_Access_and_Quality_Longitudinal_county_2008-2022
# Purpose Agreegating GHI Data.    
# File Creator : Neha Khairnar
# ==================================================

# 📦 Step 1: Imports
# =========================
import pandas as pd
from pathlib import Path

# =========================
# 📂 Step 2: Load Data
# =========================
BASE_DIR = Path().resolve().parent
#DATA_PATH = BASE_DIR / "data" / "dataset" / "hedis_primary_care.csv"
#df = pd.read_csv(DATA_PATH)

#df = pd.read_csv("../data/Datasets/SCHARE_Primary_Care_Access_and_Quality_Longitudinal_county_2008-2022/HealthCareAccessAndQuality_county_hedis.csv", nrows=10)
df = pd.read_csv("../data/Datasets/SCHARE_Primary_Care_Access_and_Quality_Longitudinal_county_2008-2022/HealthCareAccessAndQuality_county_hedis.csv")

print("Raw Data Preview:")
display(df.head(10))

# =========================
# 🧹 Step 3: Standardize Column Names
# =========================
df.columns = df.columns.str.lower()

# =========================
# 🧱 Step 4: Fix Geo Code → FIPS
# =========================
# Clean geo_code
# Convert to string + normalize
df["geo_code"] = df["geo_code"].astype(str).str.zfill(5)
#df["geo_code_5"] = df["geo_code"].astype(str).str.zfill(5)

# Split safely
df["statefips"] = df["geo_code"].str[:2]
df["countyfips"] = df["geo_code"].str[2:].str.zfill(3)

# Remove invalid codes
df = df[df["geo_code"] != "999999"]

'''
# Handle 5-digit FIPS (e.g., 55109)
mask_5 = df["geo_code"].str.len() == 5
df.loc[mask_5, "statefips"] = df.loc[mask_5, "geo_code"].str[:2]
df.loc[mask_5, "countyfips"] = df.loc[mask_5, "geo_code"].str[2:]

# Handle 4-digit codes (e.g., 8007)
mask_4 = df["geo_code"].str.len() == 4
df.loc[mask_4, "statefips"] = df.loc[mask_4, "geo_code"].str[:2]
df.loc[mask_4, "countyfips"] = df.loc[mask_4, "geo_code"].str[2:].str.zfill(3)


# Final formatting
df["statefips"] = df["statefips"].astype(str).str.zfill(2)
df["countyfips"] = df["countyfips"].astype(str).str.zfill(3)
'''


# =========================
# 📅 Step 5: Ensure Year is Numeric
# =========================
df["year"] = df["year"].astype(int)


# =========================
# 🧾 Step 6: Select Relevant Columns
# =========================
hedis = df[
    [
        "year",
        "statefips",
        "countyfips",
        "geo_code",
        "geo_label",
        "geo_name",
        "population",
        "race",
        "cohort",
        "eventname",
        "event_label",
        "observed",
        "o_e_ratio",
        "crude_rate",
        "std_error",
        "uppercl",
        "lowercl",
        'percentile',
        "adjusted_rate",
        "short_label",
        "cohort_web_label"
    ]
]

print("Cleaned HEDIS (raw level):")
display(hedis.head(10))


# ======================================================================
# 💾 Step 8: Save Clean Dataset
# =======================================================================

OUTPUT_PATH = BASE_DIR / "data" / "processed" / "neighbourhood_primary_care_access_clean.csv"

# Create folder if not exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
hedis.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ File saved to: {OUTPUT_PATH}")



'''
# =========================
# 📊 Step 7: Yearly Aggregation (County Level) Need to check how we should aggregate them yearly since we have duplicate entrith race
# =========================
hedis_yearly = (
    hedis.groupby(["year", "statefips", "countyfips"])
    .agg(
        population=("population", "sum"),
        observed=("observed", "sum"),
        o_e_ratio=("o_e_ratio", "mean"),
        crude_rate=("crude_rate", "mean"),
        adjusted_rate=("adjusted_rate", "mean"),
        std_error=("std_error", "mean"),
        uppercl=("uppercl", "mean"),
        lowercl=("lowercl", "mean")
    )
    .reset_index()
)


# =========================
# 👀 Step 8: Preview Final Dataset
# =========================
print("Yearly HEDIS Dataset:")
display(hedis_yearly.head(10))


# =========================
# 📊 Step 9: Validation
# =========================
print("Shape:", hedis_yearly.shape)
print(hedis_yearly.info())


# =========================
# ⚠️ Step 10: Missing Values Check
# =========================
print("\nMissing values:")
print(hedis_yearly.isnull().sum())


# =========================
# 💾 Step 11: Save Output
# =========================
OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

hedis_yearly.to_csv(
    OUTPUT_DIR / "hedis_yearly.csv",
    index=False
)

print("Saved to:")
print(OUTPUT_DIR / "hedis_yearly.csv")

'''


Raw Data Preview:


C:\Users\NVK00\AppData\Local\Temp\ipykernel_22404\405143952.py:20: DtypeWarning: Columns (0: Short_label, 1: cohort_web_label) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Datasets/SCHARE_Primary_Care_Access_and_Quality_Longitudinal_county_2008-2022/HealthCareAccessAndQuality_county_hedis.csv")


,Geo_code,Geo_Label,Population,Year,Race,Cohort,Eventname,Event_Label,Observed,O_E_ratio,Crude_Rate,Std_Error,Uppercl,Lowercl,Percentile,adjusted_rate,Short_label,cohort_web_label,Geo_Name
0,6083,COUNTY,3162.0,2009,Non-Black,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,2632.0,1.016315,83.238457,1.622486,86.418530,80.058383,0.667,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CA-Santa Barbara County
1,6067,COUNTY,6058.0,2010,Non-Black,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,4699.0,0.951531,77.566854,1.131549,79.784690,75.349017,0.335,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CA-Sacramento County
2,8039,COUNTY,-99999.0,2014,Black,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,-99999.0,NaN,-99999.000000,-99999.000000,-99999.000000,-99999.000000,NaN,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CO-Elbert County
3,6089,COUNTY,-99999.0,2010,Non-Black,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,-99999.0,NaN,-99999.000000,-99999.000000,-99999.000000,-99999.000000,NaN,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CA-Shasta County
4,99003,COUNTY,39.0,2014,Overall,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,32.0,1.022191,82.051282,14.504754,110.480601,53.621963,0.690,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,Other Residual ZIPs (99003)
5,8007,COUNTY,81.0,2008,Overall,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,48.0,0.739578,59.259259,8.553337,76.023800,42.494718,0.041,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CO-Archuleta County
6,6101,COUNTY,1245.0,2008,Non-Black,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,986.0,0.977642,79.196787,2.522140,84.140181,74.253394,0.467,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CA-Sutter County
7,8093,COUNTY,105.0,2015,Overall,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,78.0,0.934942,74.285714,8.411201,90.771668,57.799761,0.275,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CO-Park County
8,8089,COUNTY,-99999.0,2013,Black,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,-99999.0,NaN,-99999.000000,-99999.000000,-99999.000000,-99999.000000,NaN,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CO-Otero County
9,8091,COUNTY,31.0,2008,Overall,Diabetic Events,diab_bloodlip,Percent of Diabetic Medicare Enrollees Receivi...,27.0,1.087001,87.096774,16.761782,119.949867,54.243681,0.949,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes,CO-Ouray County


Cleaned HEDIS (raw level):


,year,statefips,countyfips,geo_code,geo_label,geo_name,population,race,cohort,eventname,...,observed,o_e_ratio,crude_rate,std_error,uppercl,lowercl,percentile,adjusted_rate,short_label,cohort_web_label
0,2009,06,083,06083,COUNTY,CA-Santa Barbara County,3162.0,Non-Black,Diabetic Events,diab_bloodlip,...,2632.0,1.016315,83.238457,1.622486,86.418530,80.058383,0.667,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
1,2010,06,067,06067,COUNTY,CA-Sacramento County,6058.0,Non-Black,Diabetic Events,diab_bloodlip,...,4699.0,0.951531,77.566854,1.131549,79.784690,75.349017,0.335,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
2,2014,08,039,08039,COUNTY,CO-Elbert County,-99999.0,Black,Diabetic Events,diab_bloodlip,...,-99999.0,NaN,-99999.000000,-99999.000000,-99999.000000,-99999.000000,NaN,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
3,2010,06,089,06089,COUNTY,CA-Shasta County,-99999.0,Non-Black,Diabetic Events,diab_bloodlip,...,-99999.0,NaN,-99999.000000,-99999.000000,-99999.000000,-99999.000000,NaN,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
4,2014,99,003,99003,COUNTY,Other Residual ZIPs (99003),39.0,Overall,Diabetic Events,diab_bloodlip,...,32.0,1.022191,82.051282,14.504754,110.480601,53.621963,0.690,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
5,2008,08,007,08007,COUNTY,CO-Archuleta County,81.0,Overall,Diabetic Events,diab_bloodlip,...,48.0,0.739578,59.259259,8.553337,76.023800,42.494718,0.041,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
6,2008,06,101,06101,COUNTY,CA-Sutter County,1245.0,Non-Black,Diabetic Events,diab_bloodlip,...,986.0,0.977642,79.196787,2.522140,84.140181,74.253394,0.467,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
7,2015,08,093,08093,COUNTY,CO-Park County,105.0,Overall,Diabetic Events,diab_bloodlip,...,78.0,0.934942,74.285714,8.411201,90.771668,57.799761,0.275,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
8,2013,08,089,08089,COUNTY,CO-Otero County,-99999.0,Black,Diabetic Events,diab_bloodlip,...,-99999.0,NaN,-99999.000000,-99999.000000,-99999.000000,-99999.000000,NaN,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes
9,2008,08,091,08091,COUNTY,CO-Ouray County,31.0,Overall,Diabetic Events,diab_bloodlip,...,27.0,1.087001,87.096774,16.761782,119.949867,54.243681,0.949,NaN,Blood Lipids Testing for Diabetics,Secondary Screening for Diabetes



✅ File saved to: C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed\neighbourhood_primary_care_access_clean.csv
